# R1-05 LangSmith monitoring sample
Makes the AI triage step visible: every draft is traced in LangSmith with input, output, time, tokens and cost, and scored by simple code checks (no AI grading).

**Inputs (not changed by this notebook):** `knowledgebase/` PDFs 01-03 and `kb_compiled.md`, `n8n/triage_prompt.md` (version read from the file, now v3), `n8n/workflow.json`, `data/processed/tobi_tuning_150.csv`, `data/processed/tobi_test_50.csv`.

**Creates:** `data/processed/synthetic_policy_cases.csv` (8 synthetic Spreekorb emails), `langsmith/heldout_priority_labels.csv` (for your Policy 8 priority labels), `langsmith/human_review.csv`, `langsmith/trace_summary.csv` (feeds M7 in R1-03 and R1-06), `langsmith/monitoring_note.md`. In LangSmith: datasets `spreekorb-dev-tuning-10`, `spreekorb-dev-policy-8` and `spreekorb-heldout-10`, experiments `dev-<version>-*` and `heldout-<version>-*` (now v3).

**How to run (Ayhan):**
1. Run all cells with `PROMPT_FROZEN = False`. This makes the dev runs (10 tuning emails + 8 policy cases) and stops before the held-out set.
2. Read the dev tables. Freeze the prompt, or change `n8n/triage_prompt.md` (bump the version, update the n8n node) and run again.
3. Fill `priority_human` in `langsmith/heldout_priority_labels.csv` (one of very_low, low, medium, high, critical, per Spreekorb Policy 8). Do not look at AI outputs while labelling.
4. Set `PROMPT_FROZEN = True` and run all again. The held-out run happens **once** per prompt version (guard) and only when all 10 labels are filled. Set `RUN_DEV = False` then, so no new dev experiment is made.
5. Fill `score_1_5` in `langsmith/human_review.csv` (note column: no names of people), set `UPLOAD_HUMAN_SCORES = True`, run the upload cell.
6. USD_TO_EUR is set (0.9123, tradingview.com, 27 Nov 2026). Add screenshots to `langsmith/screenshots/` and the share link, run the last cells again.

Data: Tobi-Bueck tickets and the 8 policy cases are **synthetic**. No real customer data, no agent names or IDs.

Ground truth: queue = Tobi queue label. Priority (official) = your human label per Spreekorb Policy 8; the Tobi priority is synthetic and noisy and is reported as a secondary column (`tobi_priority_match`).

In [1]:
# 0 Settings and 1 Check: right folder, input files exist, keys present (values are never printed)
from pathlib import Path
import subprocess, shutil, datetime, hashlib, re, json, time
import pandas as pd
from dotenv import dotenv_values, load_dotenv

_top = subprocess.run(['git', 'rev-parse', '--show-toplevel'], capture_output=True, text=True).stdout.strip()
REPO = Path(_top) if _top else Path.cwd().parent
# the folder is called knowledgebase/ in this repo (triage_prompt.md writes knowledge_base/)
KB = next((REPO / n for n in ['knowledgebase', 'knowledge_base'] if (REPO / n).exists()), REPO / 'knowledgebase')
PROC = REPO / 'data' / 'processed'
LS = REPO / 'langsmith'
NOTEBOOK = REPO / 'notebooks' / 'R1-05_langsmith.ipynb'

# ---- switches for Ayhan
RUN_DEV = True                # dev run on 10 tuning emails (each run makes a new dev experiment)
PROMPT_FROZEN = False         # set True after the STOP cell; then the held-out run happens once
UPLOAD_HUMAN_SCORES = False   # set True after you filled score_1_5 in langsmith/human_review.csv
RUN_UNDO = False              # True restores the backup of the langsmith/ output files

# ---- prices: only used if LangSmith shows no cost. Fill from https://openai.com/api/pricing (gpt-4o-mini)
PRICE_IN_USD_PER_1M = None    # USD per 1M input tokens
PRICE_OUT_USD_PER_1M = None   # USD per 1M output tokens
# ---- USD to EUR for the R1-06 handoff (LangSmith cost is in USD). Fill a dated rate and its source.
USD_TO_EUR = 0.9123           # 1 USD = 0.9123 EUR, given by Ayhan on 27 Nov 2026
USD_TO_EUR_SOURCE = 'tradingview.com (USDEUR), 27 Nov 2026'
LANGSMITH_SHARE_URL = 'PENDING: paste the public share link of the held-out experiment'

# ---- names
MODEL = 'gpt-4o-mini'
TEMPERATURE = 0.2
TRACE_NAME = 'spreekorb_triage'
DEV_DATASET = 'spreekorb-dev-tuning-10-v2'   # v2: new e-commerce sample from R1-02 fix
POLICY_DATASET = 'spreekorb-dev-policy-8'
HELDOUT_DATASET = 'spreekorb-heldout-10-v2'   # v2: new e-commerce sample from R1-02 fix
SEED = 42

IN = {
    'pdf01': KB / '01_customer_service_policy.pdf',
    'pdf02': KB / '02_brand_voice_guide.pdf',
    'pdf03': KB / '03_reply_guardrails.pdf',
    'kb_compiled': KB / 'kb_compiled.md',
    'compile_kb': KB / 'compile_kb.py',
    'prompt': REPO / 'n8n' / 'triage_prompt.md',
    'workflow': REPO / 'n8n' / 'workflow.json',
    'tuning': PROC / 'tobi_tuning_150.csv',
    'test': PROC / 'tobi_test_50.csv',
    'requirements': REPO / 'requirements.txt',
}
# every file this notebook creates or changes
OUT = {
    'policy': PROC / 'synthetic_policy_cases.csv',
    'labels': LS / 'heldout_priority_labels.csv',
    'human': LS / 'human_review.csv',
    'trace': LS / 'trace_summary.csv',
    'note': LS / 'monitoring_note.md',
}

results = []
def check(name, ok, detail=''):
    # one PASS/FAIL line, kept in results for the final count
    results.append((name, bool(ok)))
    print(('PASS' if ok else 'FAIL') + ' | ' + name + (' | ' + str(detail) if detail else ''))

check('right folder (repo root has AGENTS.md)', (REPO / 'AGENTS.md').exists(), REPO.name)
for key, p in IN.items():
    check('input exists: ' + str(p.relative_to(REPO)), p.exists())

# .env: presence only. Values are never printed or kept.
ENV_FILE = REPO / '.env'
check('.env file exists', ENV_FILE.exists())
_env = dotenv_values(ENV_FILE) if ENV_FILE.exists() else {}
for k in ['OPENAI_API_KEY', 'LANGSMITH_API_KEY']:
    check(k + ' is set (value not shown)', bool((_env.get(k) or '').strip()))
check('LANGSMITH_TRACING=true', (_env.get('LANGSMITH_TRACING') or '').strip().lower() == 'true')
del _env
load_dotenv(ENV_FILE, override=False)

# requirements.txt: package names only
_req = {re.split(r'[<>=~!\[ ;]', l.strip())[0].lower() for l in IN['requirements'].read_text().splitlines() if l.strip() and not l.strip().startswith('#')}
for pkg in ['langsmith', 'openai', 'pypdf', 'reportlab']:
    check('requirements.txt has ' + pkg, pkg in _req, '' if pkg in _req else 'missing: Ayhan adds it (requirements.txt is not owned by R1-05)')

PROMPT_MD = IN['prompt'].read_text(encoding='utf-8')
PROMPT_VERSION = re.search(r'Version:\s*(v\d+)', PROMPT_MD).group(1)
print('Prompt version:', PROMPT_VERSION)

PASS | right folder (repo root has AGENTS.md) | ecommerce_customersatisfaction_adiadoption
PASS | input exists: knowledgebase/01_customer_service_policy.pdf
PASS | input exists: knowledgebase/02_brand_voice_guide.pdf
PASS | input exists: knowledgebase/03_reply_guardrails.pdf
PASS | input exists: knowledgebase/kb_compiled.md
PASS | input exists: knowledgebase/compile_kb.py
PASS | input exists: n8n/triage_prompt.md
PASS | input exists: n8n/workflow.json
PASS | input exists: data/processed/tobi_tuning_150.csv
PASS | input exists: data/processed/tobi_test_50.csv
PASS | input exists: requirements.txt
PASS | .env file exists
PASS | OPENAI_API_KEY is set (value not shown)
PASS | LANGSMITH_API_KEY is set (value not shown)
PASS | LANGSMITH_TRACING=true
PASS | requirements.txt has langsmith
PASS | requirements.txt has openai
PASS | requirements.txt has pypdf
PASS | requirements.txt has reportlab
Prompt version: v3


In [2]:
# 2 Backup: copy every output file that already exists to _backup/<timestamp>/, and hash files we do not own
TS = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
BK = REPO / '_backup' / TS
existed = {}
for key, p in OUT.items():
    existed[key] = p.exists()
    if p.exists():
        dest = BK / p.relative_to(REPO)
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(p, dest)
print('Already existing (backed up):', [k for k, v in existed.items() if v] or 'none')
print('Backup folder:', BK)

# fingerprint of files this card must not change (.env is never read here)
WATCH = ['knowledgebase', 'knowledge_base', 'n8n', 'data/processed', 'specs', 'research', 'dashboard', 'notebooks', 'langsmith/screenshots']
WATCH_FILES = ['AGENTS.md', 'README.md', 'board.md', 'requirements.txt', 'CLAUDE.md']
def snapshot():
    files = [REPO / f for f in WATCH_FILES if (REPO / f).exists()]
    for d in WATCH:
        if (REPO / d).exists():
            files += [p for p in (REPO / d).rglob('*') if p.is_file()]
    # our own outputs (e.g. data/processed/synthetic_policy_cases.csv) are not in the fingerprint
    skip = lambda p: p == NOTEBOOK or p in OUT.values() or '.ipynb_checkpoints' in p.parts or p.name == '.DS_Store'
    return {str(p.relative_to(REPO)): hashlib.sha256(p.read_bytes()).hexdigest() for p in files if not skip(p)}
BEFORE = snapshot()
print('Files fingerprinted:', len(BEFORE))

Already existing (backed up): none
Backup folder: /Users/macbook-air/Desktop/AI consulting/Week8/Day4/ecommerce_customersatisfaction_adiadoption/_backup/20261127_112551
Files fingerprinted: 52


In [3]:
# 3a Ingest: extract PDFs 01-03 with the same logic as knowledgebase/compile_kb.py (nothing is written)
from pypdf import PdfReader

FOOT = 'Invented for an Ironhack case study. Spreekorb is not a real company. Not legal advice.'
DOCS = ['01_customer_service_policy.pdf', '02_brand_voice_guide.pdf', '03_reply_guardrails.pdf']

def compile_kb(folder):
    parts = []
    for name in DOCS:
        text = '\n'.join(page.extract_text() for page in PdfReader(folder / name).pages)
        lines = [l.strip() for l in text.split('\n')]
        # drop empty lines, the footer and page numbers
        lines = [l for l in lines if l and FOOT not in l and not re.fullmatch(r'Page \d+', l)]
        # a lone bullet glyph becomes "- " in front of the next line
        clean, bullet = [], False
        for l in lines:
            if len(l) == 1 and not l.isalnum():
                bullet = True
                continue
            clean.append(('- ' if bullet else '') + l)
            bullet = False
        parts.append('=== ' + name.replace('.pdf', '') + ' ===\n' + '\n'.join(clean))
    return '\n\n'.join(parts) + '\n'

KB_TEXT = compile_kb(KB)
INGEST_OK = KB_TEXT == IN['kb_compiled'].read_text(encoding='utf-8')
check('ingest: PDFs 01-03 give exactly kb_compiled.md', INGEST_OK, str(len(KB_TEXT.split())) + ' words, sha256 ' + hashlib.sha256(KB_TEXT.encode()).hexdigest()[:12])

PASS | ingest: PDFs 01-03 give exactly kb_compiled.md | 1423 words, sha256 a8b7e964b24c


In [4]:
# 3b System prompt: SYSTEM section of triage_prompt.md with {knowledge_base} filled, compared with n8n
SYSTEM_TEMPLATE = PROMPT_MD.split('## SYSTEM\n', 1)[1].split('\n## USER', 1)[0].strip()
USER_TEMPLATE = PROMPT_MD.split('## USER\n', 1)[1].strip()
SYSTEM_PROMPT = SYSTEM_TEMPLATE.replace('{knowledge_base}', KB_TEXT.strip())

WF = json.loads(IN['workflow'].read_text(encoding='utf-8'))
node = lambda name: next(n for n in WF['nodes'] if n['name'] == name)
N8N_SYSTEM = node('Triage + draft')['parameters']['messages']['messageValues'][0]['message']
N8N_USER = node('Triage + draft')['parameters']['text']
N8N_LLM = node('OpenAI gpt-4o-mini')['parameters']
SCHEMA = json.loads(node('Output format')['parameters']['inputSchema'])

PROMPT_OK = SYSTEM_PROMPT == N8N_SYSTEM
check('system prompt equals the n8n "Triage + draft" system message', PROMPT_OK, str(len(SYSTEM_PROMPT)) + ' characters')
# same user message as n8n: "=Subject: {{ $json.subject }}" becomes "Subject: {subject}"
n8n_user = N8N_USER.lstrip('=').replace('{{ $json.subject }}', '{subject}').replace('{{ $json.body }}', '{body}')
check('user message template equals n8n', USER_TEMPLATE == n8n_user, USER_TEMPLATE.replace('\n', ' / '))
check('n8n uses the same model and temperature', N8N_LLM['model']['value'] == MODEL and N8N_LLM['options']['temperature'] == TEMPERATURE)

PASS | system prompt equals the n8n "Triage + draft" system message | 11389 characters
PASS | user message template equals n8n | Subject: {subject} / Email: {body}
PASS | n8n uses the same model and temperature


In [5]:
# 3c Target function: wrapped OpenAI client, gpt-4o-mini, temperature 0.2, JSON output with the n8n schema
from openai import OpenAI
from langsmith import Client, traceable
from langsmith.wrappers import wrap_openai

client = Client()               # reads LANGSMITH_API_KEY from the environment
oai = wrap_openai(OpenAI())     # every OpenAI call is traced (tokens, cost, time)

FIELDS = ['queue', 'priority', 'reason', 'draft_reply', 'policy_sections']
check('schema from workflow.json has the 5 fields, all required', list(SCHEMA['properties']) == FIELDS and SCHEMA['required'] == FIELDS)
# strict JSON mode needs additionalProperties false; fields, types and enums stay the n8n ones
RESPONSE_FORMAT = {'type': 'json_schema', 'json_schema': {'name': 'spreekorb_triage', 'strict': True, 'schema': {**SCHEMA, 'additionalProperties': False}}}

@traceable(name=TRACE_NAME, metadata={'prompt_version': PROMPT_VERSION, 'model': MODEL, 'temperature': TEMPERATURE})
def triage(inputs: dict) -> dict:
    # same cleaning as the n8n "Clean email" node: trim subject and body
    user = USER_TEMPLATE.replace('{subject}', (inputs.get('subject') or '').strip()).replace('{body}', (inputs.get('body') or '').strip())
    resp = oai.chat.completions.create(
        model=MODEL,
        temperature=TEMPERATURE,
        response_format=RESPONSE_FORMAT,
        messages=[{'role': 'system', 'content': SYSTEM_PROMPT}, {'role': 'user', 'content': user}],
    )
    return json.loads(resp.choices[0].message.content)

print('Target ready:', TRACE_NAME, '|', MODEL, '| temperature', TEMPERATURE, '| prompt', PROMPT_VERSION)

PASS | schema from workflow.json has the 5 fields, all required
Target ready: spreekorb_triage | gpt-4o-mini | temperature 0.2 | prompt v3


In [6]:
# 3d Code evaluators only (no AI grading). Each returns a score 1 (ok) or 0 (not ok).
PRIORITIES = ['very_low', 'low', 'medium', 'high', 'critical']

# banned phrases come from the guardrails table in the knowledge base (EN + DE), plus the em dash
_table = KB_TEXT.split('=== 03_reply_guardrails ===')[1].split('English\nGerman\n', 1)[1].split('Style bans:', 1)[0]
BANNED = []
for row in _table.strip().split('\n'):
    row = re.sub(r'\(.*?\)', '', row)              # "(without an alternative)" is a condition, not a phrase
    for piece in row.split(' / '):
        words = [w.strip() for w in piece.split(', ')]
        # "Seamless, delve, leverage" is a word list; "Ich hoffe, diese E-Mail ..." is one phrase
        BANNED += words if all(' ' not in w for w in words) else [piece.strip()]
BANNED = [b for b in BANNED if b] + ['—']   # — is the em dash
print(len(BANNED), 'banned phrases:', BANNED)
check('banned phrase list parsed (EN + DE + em dash)', len(BANNED) >= 25 and 'Rest assured' in BANNED and 'Wie bereits erwähnt' in BANNED, str(len(BANNED)) + ' phrases')

def banned_hits(text):
    t = ' '.join(text.split()).lower()
    return [b for b in BANNED if re.search(r'(?<!\w)' + re.escape(b.lower()) + r'(?!\w)', t) or (b == '—' and b in t)]

# a policy fact = a number of days, working days or hours, or a EUR amount
FACT_RE = re.compile(r'\d+\s*(?:(?:to|bis|-)\s*\d+\s*)?(?:working days|days|day|werktage\w*|tage\w*|tagen|hours|stunden)\b|\beur\b|€', re.I)
SECTION_RE = re.compile(r'Policy ([1-9]|10)')
DE_WORDS = {'und', 'der', 'die', 'das', 'du', 'dein', 'deine', 'dich', 'dir', 'ist', 'nicht', 'wir', 'uns', 'mit', 'für', 'zu', 'ich', 'auf', 'bei', 'hallo', 'viele', 'grüße', 'kundenservice', 'eine', 'ein', 'sobald', 'kannst'}
EN_WORDS = {'the', 'and', 'you', 'your', 'is', 'we', 'us', 'to', 'for', 'with', 'of', 'this', 'hi', 'best', 'wishes', 'customer', 'service', 'can', 'will', 'our', 'please'}

def guess_language(text):
    # simple word count: more German function words means German
    words = re.findall(r'[a-zäöüß]+', text.lower())
    de, en = sum(w in DE_WORDS for w in words), sum(w in EN_WORDS for w in words)
    return 'de' if de > en else 'en'

def _draft(o):
    return (o or {}).get('draft_reply') or ''

def queue_match(outputs, reference_outputs):
    return {'key': 'queue_match', 'score': int((outputs or {}).get('queue') == reference_outputs['queue'])}

def priority_match(outputs, reference_outputs):
    return {'key': 'priority_match', 'score': int((outputs or {}).get('priority') == reference_outputs['priority'])}

def priority_within_one(outputs, reference_outputs):
    p, r = (outputs or {}).get('priority'), reference_outputs['priority']
    ok = p in PRIORITIES and abs(PRIORITIES.index(p) - PRIORITIES.index(r)) <= 1
    return {'key': 'priority_within_one', 'score': int(ok)}

def no_banned_phrase(outputs):
    hits = banned_hits(_draft(outputs))
    return {'key': 'no_banned_phrase', 'score': int(not hits), 'comment': ', '.join(hits) or 'none'}

def cited_sections_valid(outputs):
    secs = (outputs or {}).get('policy_sections') or []
    bad = [s for s in secs if not SECTION_RE.fullmatch(str(s).strip())]
    return {'key': 'cited_sections_valid', 'score': int(not bad), 'comment': ', '.join(map(str, bad)) or 'ok'}

def policy_fact_cited(outputs):
    facts = FACT_RE.findall(_draft(outputs))
    ok = not facts or bool((outputs or {}).get('policy_sections'))
    return {'key': 'policy_fact_cited', 'score': int(ok), 'comment': 'facts: ' + str(len(facts))}

def language_match(outputs, reference_outputs):
    lang = guess_language(_draft(outputs))
    return {'key': 'language_match', 'score': int(lang == reference_outputs['language']), 'comment': 'draft looks ' + lang}

def is_escalation(outputs):
    return ((outputs or {}).get('reason') or '').strip().startswith('Escalate to')

def word_count_60_120(outputs):
    # holding replies for escalations (reason starts with "Escalate to"): 40 to 80 words, else 60 to 120
    n = len(_draft(outputs).split())
    lo, hi = (40, 80) if is_escalation(outputs) else (60, 120)
    return {'key': 'word_count_60_120', 'score': int(lo <= n <= hi), 'comment': str(n) + ' words, allowed ' + str(lo) + '-' + str(hi)}

def tobi_priority_match(outputs, reference_outputs):
    # secondary: compare with the synthetic Tobi priority (official priority = human label)
    return {'key': 'tobi_priority_match', 'score': int((outputs or {}).get('priority') == reference_outputs.get('tobi_priority'))}

def citations_used(outputs):
    # 1 if the draft cites at least one policy section
    return {'key': 'citations_used', 'score': int(bool((outputs or {}).get('policy_sections'))), 'comment': 'escalation' if is_escalation(outputs) else 'normal'}

EVALUATORS = [queue_match, priority_match, priority_within_one, no_banned_phrase, cited_sections_valid, policy_fact_cited, language_match, word_count_60_120]
EVAL_KEYS = [f.__name__ for f in EVALUATORS]
# held-out: same 8 + the secondary Tobi priority column
HELD_EVALUATORS = EVALUATORS + [tobi_priority_match]
HELD_KEYS = [f.__name__ for f in HELD_EVALUATORS]
# policy cases have no queue/priority labels (blank), so only the label-free checks + citations_used
POLICY_EVALUATORS = [no_banned_phrase, cited_sections_valid, policy_fact_cited, language_match, word_count_60_120, citations_used]
POLICY_KEYS = [f.__name__ for f in POLICY_EVALUATORS]

# quick self-test on made-up text (no API call)
_good = {'queue': 'Billing and Payments', 'priority': 'high', 'policy_sections': ['Policy 5'], 'draft_reply': 'Hi [first name], your refund goes back within 14 days. ' + 'word ' * 50}
_bad = {'queue': 'Billing and Payments', 'priority': 'low', 'policy_sections': [], 'draft_reply': 'Rest assured — refund in 14 days.'}
_ref = {'queue': 'Billing and Payments', 'priority': 'high', 'language': 'en'}
check('evaluator self-test: good example passes', no_banned_phrase(_good)['score'] and policy_fact_cited(_good)['score'] and priority_within_one(_good, _ref)['score'] and word_count_60_120(_good)['score'])
check('evaluator self-test: bad example fails', not no_banned_phrase(_bad)['score'] and not policy_fact_cited(_bad)['score'] and not priority_within_one(_bad, _ref)['score'] and not word_count_60_120(_bad)['score'])
check('evaluator self-test: Policy 11 is not a valid section', cited_sections_valid({'policy_sections': ['Policy 11']})['score'] == 0 and cited_sections_valid({'policy_sections': ['Policy 10']})['score'] == 1)
_esc = {'reason': 'Escalate to Data Protection Officer: deletion request.', 'draft_reply': 'word ' * 50}
check('evaluator self-test: 50 words ok for escalation, not for a normal reply', word_count_60_120(_esc)['score'] == 1 and word_count_60_120({'reason': 'Normal.', 'draft_reply': 'word ' * 50})['score'] == 0)

29 banned phrases: ['I hope this email finds you well', 'Ich hoffe, diese E-Mail erreicht dich gut', 'Rest assured', 'Seien Sie versichert', 'We apologize for any inconvenience caused', 'Wir entschuldigen uns für etwaige Unannehmlichkeiten', 'Thank you for reaching out to us', 'Vielen Dank, dass Sie sich an uns gewandt haben', 'Seamless', 'delve', 'leverage', 'elevate', 'nahtlos', 'maßgeschneidert', 'ganzheitlich', 'As I already said', 'As mentioned before', 'Wie bereits erwähnt', 'Unfortunately that is not possible', 'Leider nicht möglich', 'Your fault', 'you should have', 'Ihr Fehler', 'Sie hätten', 'We guarantee', '100%', 'Wir garantieren', '100 %', '—']
PASS | banned phrase list parsed (EN + DE + em dash) | 29 phrases
PASS | evaluator self-test: good example passes
PASS | evaluator self-test: bad example fails
PASS | evaluator self-test: Policy 11 is not a valid section
PASS | evaluator self-test: 50 words ok for escalation, not for a normal reply


In [7]:
# 3e Datasets: 10 dev emails (tuning file, seed 42) and the 10 held-out timing emails (test file)
tu = pd.read_csv(IN['tuning'])
te = pd.read_csv(IN['test'])
te['timing_test'] = te['timing_test'].fillna('').astype(str).str.lower().eq('true')

# dev: one email per queue x language (8), then 2 more at random, all with seed 42
_first = tu.groupby(['queue', 'language']).sample(1, random_state=SEED)
_rest = tu.drop(_first.index).sample(10 - len(_first), random_state=SEED)
dev_df = pd.concat([_first, _rest]).sort_values('ticket_id').reset_index(drop=True)
held_df = te[te['timing_test']].sort_values('ticket_id').reset_index(drop=True)

check('dev: 10 emails, all 4 queues, de and en', len(dev_df) == 10 and dev_df['queue'].nunique() == 4 and set(dev_df['language']) == {'de', 'en'}, dev_df['queue'].value_counts().to_dict())
check('held-out: the 10 timing_test emails from the test file', len(held_df) == 10, ', '.join(held_df['ticket_id']))
check('no overlap between dev and held-out', not set(dev_df['ticket_id']) & set(held_df['ticket_id']))

def make_examples(df):
    # inputs = what the model sees; outputs = labels for the evaluators; text fields cleaned the safe way (pandas 3)
    ex = []
    for _, r in df.iterrows():
        ex.append({
            'inputs': {'email_id': r['ticket_id'], 'subject': str(r['subject'] if pd.notna(r['subject']) else '').strip(), 'body': str(r['body'] if pd.notna(r['body']) else '').strip()},
            'outputs': {'queue': r['queue'], 'priority': r['priority'], 'language': r['language']},
            'metadata': {'email_id': r['ticket_id'], 'split': r['split'], 'source': 'Tobi-Bueck (synthetic)'},
        })
    return ex

def get_or_create_dataset(name, examples, description):
    # reuse the dataset if it exists, but only if it holds exactly the same emails
    want = sorted(e['inputs']['email_id'] for e in examples)
    if client.has_dataset(dataset_name=name):
        ds = client.read_dataset(dataset_name=name)
        ids = sorted((e.metadata or {}).get('email_id') for e in client.list_examples(dataset_id=ds.id))
        check(name + ': existing dataset has the same ' + str(len(want)) + ' emails', ids == want, ids)
        return ds
    ds = client.create_dataset(name, description=description)
    client.create_examples(dataset_id=ds.id, examples=examples)
    check(name + ': dataset created with ' + str(len(want)) + ' emails', len(list(client.list_examples(dataset_id=ds.id))) == len(want))
    return ds

dev_ds = get_or_create_dataset(DEV_DATASET, make_examples(dev_df), 'R1-05 dev: 10 Tobi tuning emails (synthetic), seed 42, mix of queues and de/en')
# the held-out dataset is created in 3g, only after your priority_human labels are filled

def experiments(dataset, prefix):
    # experiments of this dataset whose name starts with the prefix, newest first (none if the dataset does not exist yet)
    if not client.has_dataset(dataset_name=dataset):
        return []
    ps = [p for p in client.list_projects(reference_dataset_name=dataset) if p.name.startswith(prefix + '-')]
    return sorted(ps, key=lambda p: p.start_time, reverse=True)

def load_experiment(name, n_expected=10, n_keys=len(EVAL_KEYS), tries=12):
    # read the experiment back from LangSmith: root runs (time, input, output), LLM runs (tokens, cost), feedback (scores)
    for _ in range(tries):
        roots = [r for r in client.list_runs(project_name=name, is_root=True) if r.name == TRACE_NAME]
        llms = list(client.list_runs(project_name=name, run_type='llm'))
        fb = list(client.list_feedback(run_ids=[r.id for r in roots])) if roots else []
        ready = len(roots) >= n_expected and all(r.end_time for r in roots) and len(llms) >= n_expected and len(fb) >= n_expected * n_keys
        if ready:
            break
        time.sleep(10)   # LangSmith needs a few seconds to process tokens, cost and scores
    by_trace = {}
    for r in llms:
        d = by_trace.setdefault(r.trace_id, {'tokens_in': 0, 'tokens_out': 0, 'cost_usd': 0.0, 'cost_missing': False})
        d['tokens_in'] += r.prompt_tokens or 0
        d['tokens_out'] += r.completion_tokens or 0
        if r.total_cost is None:
            d['cost_missing'] = True
        else:
            d['cost_usd'] += float(r.total_cost)
    rows = []
    for r in roots:
        inp = (r.inputs or {}).get('inputs', r.inputs or {})
        out = r.outputs or {}
        u = by_trace.get(r.trace_id, {'tokens_in': None, 'tokens_out': None, 'cost_usd': None, 'cost_missing': True})
        rows.append({
            'email_id': inp.get('email_id'), 'run_id': str(r.id), 'error': r.error,
            'queue_ai': out.get('queue'), 'priority_ai': out.get('priority'), 'reason': out.get('reason'),
            'draft_reply': out.get('draft_reply'), 'policy_sections': ', '.join(out.get('policy_sections') or []),
            'latency_s': (r.end_time - r.start_time).total_seconds() if r.end_time else None,
            'tokens_in': u['tokens_in'], 'tokens_out': u['tokens_out'],
            'cost_usd': None if u['cost_missing'] else u['cost_usd'],
        })
    df = pd.DataFrame(rows)
    if df.empty:
        return df
    scores = pd.DataFrame([{'run_id': str(f.run_id), 'key': f.key, 'score': f.score} for f in fb])
    if not scores.empty:
        df = df.merge(scores.pivot_table(index='run_id', columns='key', values='score', aggfunc='last').reset_index(), on='run_id', how='left')
    return df.sort_values('email_id').reset_index(drop=True)

PASS | dev: 10 emails, all 4 queues, de and en | {'Customer Service': 4, 'Product Support': 2, 'Billing and Payments': 2, 'Returns and Exchanges': 2}
PASS | held-out: the 10 timing_test emails from the test file | T164, T168, T170, T176, T177, T181, T183, T190, T196, T199
PASS | no overlap between dev and held-out
PASS | spreekorb-dev-tuning-10: existing dataset has the same 10 emails | ['T015', 'T019', 'T022', 'T066', 'T114', 'T115', 'T118', 'T124', 'T139', 'T148']


In [8]:
# 3e2 Label sheets: 8 synthetic policy cases (data/processed) and the held-out priority sheet for Ayhan (langsmith/)
# Policy cases: invented Spreekorb emails, no real people. queue_human / priority_human stay blank for Ayhan.
POLICY_CASES = [
    ('P01', 'de', 'Rücksendung Jacke', 'Hallo, ich habe letzte Woche eine Jacke bestellt (Bestellung SPK-104233). Sie passt leider nicht. Kann ich sie zurückschicken und wie geht das?'),
    ('P02', 'en', 'Refund not received', 'Hi, I returned a pair of shoes (order SPK-104871) two weeks ago. Tracking says the parcel reached your warehouse 6 days ago, but my PayPal refund has not arrived. When will it come?'),
    ('P03', 'de', 'Paket kommt nicht an', 'Hallo, meine Bestellung SPK-105002 wurde vor 7 Werktagen verschickt, aber die DHL-Sendungsverfolgung zeigt seit 6 Tagen kein Update. Wo ist mein Paket?'),
    ('P04', 'en', 'Lamp arrived broken', 'Hello, the table lamp from order SPK-105118 arrived today with a cracked glass shade, and the box was dented. What can you do?'),
    ('P05', 'de', 'Bestellung stornieren', 'Hallo, ich habe heute Morgen aus Versehen zweimal dieselbe Bettwäsche bestellt (SPK-105230). Bitte storniert die zweite Bestellung, sie ist noch nicht verschickt.'),
    ('P06', 'en', 'Invoice needed again', 'Hi, I need the invoice for order SPK-105344 for my tax return, but I cannot find the email any more. Could you send it again?'),
    ('P07', 'de', 'Löschung meiner Daten', 'Hallo, bitte löscht mein Kundenkonto und alle Daten, die ihr über mich gespeichert habt, nach Art. 17 DSGVO, und bestätigt mir die Löschung.'),
    ('P08', 'en', 'Final warning before legal action', 'I have waited four weeks for the refund of my cancelled order SPK-105461. If the money is not in my account by Friday, my lawyer will take legal action against Spreekorb.'),
]
POLICY_COLS = ['email_id', 'subject', 'body', 'language', 'synthetic', 'queue_human', 'priority_human']
if OUT['policy'].exists():
    print('Kept existing', OUT['policy'].relative_to(REPO), '(may hold your labels).')
else:
    pd.DataFrame([{'email_id': i, 'subject': s, 'body': b, 'language': l, 'synthetic': True, 'queue_human': '', 'priority_human': ''} for i, l, s, b in POLICY_CASES])[POLICY_COLS].to_csv(OUT['policy'], index=False)
    print('Wrote', OUT['policy'].relative_to(REPO), len(POLICY_CASES), 'rows')
pol_df = pd.read_csv(OUT['policy'], dtype=str).fillna('')

# held-out priority sheet: only the email and the Tobi labels, NO AI output (so the label is not biased)
LABEL_COLS = ['email_id', 'subject', 'body', 'tobi_queue', 'tobi_priority', 'priority_human', 'note', 'queue_human']
def labels_filled(p):
    return p.exists() and pd.read_csv(p, dtype=str).get('priority_human', pd.Series(dtype=str)).fillna('').str.strip().ne('').any()
if OUT['labels'].exists():   # R1-02 writes the 50-row sheet; never overwrite it here
    print('Kept existing', OUT['labels'].relative_to(REPO), '(written by R1-02, may hold your labels).')
else:
    LS.mkdir(parents=True, exist_ok=True)
    lab = pd.DataFrame({
        'email_id': held_df['ticket_id'], 'subject': held_df['subject'].fillna('').astype(str).str.strip(),
        'body': held_df['body'].fillna('').astype(str).str.strip(), 'tobi_queue': held_df['queue'],
        'tobi_priority': held_df['priority'], 'priority_human': '', 'note': '', 'queue_human': '',
    })
    lab[LABEL_COLS].to_csv(OUT['labels'], index=False)
    print('Wrote', OUT['labels'].relative_to(REPO), len(lab), 'rows. Fill priority_human per Policy 8 (very_low, low, medium, high, critical).')

# policy dev dataset: inputs = the email; outputs = language (+ blank human labels); no queue/priority checks
pol_examples = [{
    'inputs': {'email_id': r['email_id'], 'subject': r['subject'].strip(), 'body': r['body'].strip()},
    'outputs': {'language': r['language'], 'queue_human': r['queue_human'], 'priority_human': r['priority_human']},
    'metadata': {'email_id': r['email_id'], 'split': 'dev-policy', 'source': 'synthetic Spreekorb policy case (R1-05)'},
} for _, r in pol_df.iterrows()]
pol_ds = get_or_create_dataset(POLICY_DATASET, pol_examples, 'R1-05 dev: 8 synthetic Spreekorb policy cases (4 de, 4 en), incl. data protection and legal threat escalations')

Wrote data/processed/synthetic_policy_cases.csv 8 rows
Wrote langsmith/heldout_priority_labels.csv 10 rows. Fill priority_human per Policy 8 (very_low, low, medium, high, critical).
PASS | spreekorb-dev-policy-8: dataset created with 8 emails


In [9]:
# 3f Dev run: 10 tuning emails, experiment prefix dev-<version>. Safe to repeat while tuning.
DEV_PREFIX = 'dev-' + PROMPT_VERSION
if RUN_DEV and INGEST_OK and PROMPT_OK:
    dev_res = client.evaluate(
        triage, data=DEV_DATASET, evaluators=EVALUATORS, experiment_prefix=DEV_PREFIX,
        description='R1-05 dev run, prompt ' + PROMPT_VERSION, max_concurrency=1,   # one at a time = clean latency
        metadata={'prompt_version': PROMPT_VERSION, 'model': MODEL, 'temperature': TEMPERATURE, 'set': 'dev'},
    )
    DEV_EXP = dev_res.experiment_name
    # second dev dataset: the 8 synthetic policy cases
    pol_res = client.evaluate(
        triage, data=POLICY_DATASET, evaluators=POLICY_EVALUATORS, experiment_prefix=DEV_PREFIX,
        description='R1-05 dev run on policy cases, prompt ' + PROMPT_VERSION, max_concurrency=1,
        metadata={'prompt_version': PROMPT_VERSION, 'model': MODEL, 'temperature': TEMPERATURE, 'set': 'dev-policy'},
    )
    POL_EXP = pol_res.experiment_name
elif RUN_DEV:
    DEV_EXP = POL_EXP = None
    print('Dev run skipped: ingest or prompt check failed (fix that first).')
else:
    _old = experiments(DEV_DATASET, DEV_PREFIX)
    DEV_EXP = _old[0].name if _old else None
    _oldp = experiments(POLICY_DATASET, DEV_PREFIX)
    POL_EXP = _oldp[0].name if _oldp else None
    print('RUN_DEV = False, reusing the newest dev experiments:', DEV_EXP, POL_EXP)

dev_out = load_experiment(DEV_EXP) if DEV_EXP else pd.DataFrame()
if not dev_out.empty:
    ref = dev_df[['ticket_id', 'queue', 'priority', 'language']].rename(columns={'ticket_id': 'email_id'})
    dev_table = ref.merge(dev_out, on='email_id', how='left')
    dev_table['words'] = dev_table['draft_reply'].fillna('').astype(str).str.split().str.len()
    show = ['email_id', 'language', 'queue', 'queue_ai', 'priority', 'priority_ai', 'policy_sections', 'words', 'latency_s'] + [k for k in EVAL_KEYS if k in dev_table]
    pd.set_option('display.max_colwidth', 40)
    print('Experiment:', DEV_EXP)
    display(dev_table[show])
    print('Share of 10 passing per evaluator:')
    print(dev_table[[k for k in EVAL_KEYS if k in dev_table]].mean().round(2).to_string())

# policy cases table + citations_used (expected high, except the escalations P07 and P08)
pol_out = load_experiment(POL_EXP, n_expected=8, n_keys=len(POLICY_KEYS)) if POL_EXP else pd.DataFrame()
if not pol_out.empty:
    pol_out['escalation'] = pol_out['reason'].fillna('').astype(str).str.strip().str.startswith('Escalate to')
    pol_out['words'] = pol_out['draft_reply'].fillna('').astype(str).str.split().str.len()
    pol_table = pol_df[['email_id', 'language', 'subject']].merge(pol_out, on='email_id', how='left')
    print('Experiment:', POL_EXP)
    display(pol_table[['email_id', 'language', 'subject', 'queue_ai', 'priority_ai', 'escalation', 'policy_sections', 'words'] + [k for k in POLICY_KEYS if k in pol_table]])
    _norm = pol_out[~pol_out['escalation']]
    CITE_SHARE = round(_norm['citations_used'].mean(), 2) if ('citations_used' in pol_out and len(_norm)) else None
    print('citations_used, share of non-escalation drafts with policy_sections:', CITE_SHARE, '(' + str(len(_norm)) + ' drafts)')
else:
    CITE_SHARE = None

/opt/anaconda3/envs/bootcamp-env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


View the evaluation results for experiment: 'dev-v3-729cd3f9' at:
https://smith.langchain.com/o/02ee3189-6ecc-4e1c-b911-6d5a92d2b83c/datasets/0cc2ff4e-f3a2-4edd-a8e5-358b98c5534a/compare?selectedSessions=f99df542-deb2-4cd7-aee3-62d8608438c3




10it [00:22,  2.25s/it]


View the evaluation results for experiment: 'dev-v3-460d301b' at:
https://smith.langchain.com/o/02ee3189-6ecc-4e1c-b911-6d5a92d2b83c/datasets/5cda4a24-fa6c-4061-8529-08cf7aeedfc0/compare?selectedSessions=785e4fb4-2a60-4b15-a85d-2e0136df7bb2




8it [00:17,  2.18s/it]
/var/folders/v2/tdj6p3nj1bz3h2g88l897pr40000gn/T/ipykernel_36009/2552493047.py:53: DeprecationWarning: list_runs() is deprecated and will be removed after Jan 31, 2027. Use client.runs.query() instead.See https://docs.langchain.com/langsmith/smithdb-sdk-migration-query-runs#runs-query for the migration guide.
  roots = [r for r in client.list_runs(project_name=name, is_root=True) if r.name == TRACE_NAME]
/var/folders/v2/tdj6p3nj1bz3h2g88l897pr40000gn/T/ipykernel_36009/2552493047.py:54: DeprecationWarning: list_runs() is deprecated and will be removed after Jan 31, 2027. Use client.runs.query() instead.See https://docs.langchain.com/langsmith/smithdb-sdk-migration-query-runs#runs-query for the migration guide.
  llms = list(client.list_runs(project_name=name, run_type='llm'))


Experiment: dev-v3-729cd3f9


... (2 outputs left out)

/var/folders/v2/tdj6p3nj1bz3h2g88l897pr40000gn/T/ipykernel_36009/2552493047.py:53: DeprecationWarning: list_runs() is deprecated and will be removed after Jan 31, 2027. Use client.runs.query() instead.See https://docs.langchain.com/langsmith/smithdb-sdk-migration-query-runs#runs-query for the migration guide.
  roots = [r for r in client.list_runs(project_name=name, is_root=True) if r.name == TRACE_NAME]
/var/folders/v2/tdj6p3nj1bz3h2g88l897pr40000gn/T/ipykernel_36009/2552493047.py:54: DeprecationWarning: list_runs() is deprecated and will be removed after Jan 31, 2027. Use client.runs.query() instead.See https://docs.langchain.com/langsmith/smithdb-sdk-migration-query-runs#runs-query for the migration guide.
  llms = list(client.list_runs(project_name=name, run_type='llm'))


Experiment: dev-v3-460d301b


,email_id,language,subject,queue_ai,priority_ai,escalation,policy_sections,words,no_banned_phrase,cited_sections_valid,policy_fact_cited,language_match,word_count_60_120,citations_used
0,P01,de,Rücksendung Jacke,Returns and Exchanges,medium,False,"Policy 4, Policy 5",67,1.0,1.0,1.0,1.0,1.0,1.0
1,P02,en,Refund not received,Billing and Payments,high,False,Policy 5,58,1.0,1.0,1.0,1.0,0.0,1.0
2,P03,de,Paket kommt nicht an,Customer Service,high,False,Policy 2,53,1.0,1.0,1.0,1.0,0.0,1.0
3,P04,en,Lamp arrived broken,Returns and Exchanges,high,False,Policy 6,48,1.0,1.0,1.0,1.0,0.0,1.0
4,P05,de,Bestellung stornieren,Customer Service,high,False,Policy 3,47,1.0,1.0,1.0,1.0,0.0,1.0
5,P06,en,Invoice needed again,Billing and Payments,medium,False,Policy 7,36,1.0,1.0,1.0,1.0,0.0,1.0
6,P07,de,Löschung meiner Daten,Customer Service,critical,True,Policy 9,33,1.0,1.0,1.0,1.0,0.0,1.0
7,P08,en,Final warning before legal action,Billing and Payments,high,False,"Policy 5, Policy 8",68,1.0,1.0,1.0,0.0,1.0,1.0


citations_used, share of non-escalation drafts with policy_sections: 1.0 (7 drafts)


## STOP
**Freeze the prompt, or change `n8n/triage_prompt.md` (bump version) and re-run dev.**

If you change the prompt: edit `triage_prompt.md` (version v3, ...), paste the new system message into the n8n "Triage + draft" node, export `workflow.json`, then run all again. The prompt check fails until both match.
Never tune on the held-out emails. When the prompt is frozen, set `PROMPT_FROZEN = True` and `RUN_DEV = False` in cell 1 and run all again.

In [ ]:
# STOP cell: read the dev table above before you go on
print('STOP: Freeze the prompt, or change triage_prompt.md (bump version) and re-run dev.')
print('Prompt version:', PROMPT_VERSION, '| PROMPT_FROZEN =', PROMPT_FROZEN)
if not PROMPT_FROZEN:
    print('The held-out run will be skipped. Set PROMPT_FROZEN = True in cell 1 when the prompt is frozen.')

In [ ]:
# 3g Official run: the 10 held-out timing emails, prefix heldout-<version>. Runs ONCE per prompt version.
HELD_PREFIX = 'heldout-' + PROMPT_VERSION
# official priority truth = Ayhan's priority_human (Policy 8); queue truth = Tobi queue; Tobi priority = secondary
labels = pd.read_csv(OUT['labels'], dtype=str).fillna('')
labels = labels[labels['email_id'].isin(held_df['ticket_id'])].reset_index(drop=True)   # sheet has 50 rows; use the 10 held-out ones
labels['priority_human'] = labels['priority_human'].str.strip()
LABELS_OK = len(labels) == 10 and labels['priority_human'].isin(PRIORITIES).all() and sorted(labels['email_id']) == sorted(held_df['ticket_id'])
human_of = dict(zip(labels['email_id'], labels['priority_human']))

def make_held_examples(df):
    return [{
        'inputs': {'email_id': r['ticket_id'], 'subject': str(r['subject'] if pd.notna(r['subject']) else '').strip(), 'body': str(r['body'] if pd.notna(r['body']) else '').strip()},
        'outputs': {'queue': r['queue'], 'priority': human_of[r['ticket_id']], 'tobi_priority': r['priority'], 'language': r['language']},
        'metadata': {'email_id': r['ticket_id'], 'split': r['split'], 'source': 'Tobi-Bueck (synthetic)', 'priority_source': 'priority_human (Ayhan, Policy 8)'},
    } for _, r in df.iterrows()]

_done = experiments(HELDOUT_DATASET, HELD_PREFIX)
if _done:
    HELD_EXP = _done[0].name
    print('Guard: held-out experiment for', PROMPT_VERSION, 'already exists (' + HELD_EXP + '). Not running again, reading it back.')
    print('If that run failed part-way, delete it in LangSmith, then run this cell again.')
elif not LABELS_OK:
    HELD_EXP = None
    print('PENDING: fill priority_human for all 10 emails in', OUT['labels'].relative_to(REPO), '(very_low, low, medium, high, critical). Held-out run stopped here.')
elif PROMPT_FROZEN and INGEST_OK and PROMPT_OK:
    held_ds = get_or_create_dataset(HELDOUT_DATASET, make_held_examples(held_df), 'R1-05 held-out: the 10 Tobi test emails flagged timing_test (synthetic). Priority truth = human label (Policy 8). Run once per prompt version.')
    # an older dataset without your labels must not be used
    _ex = {(e.metadata or {}).get('email_id'): e.outputs or {} for e in client.list_examples(dataset_id=held_ds.id)}
    DS_LABELS_OK = all(_ex.get(i, {}).get('priority') == p and 'tobi_priority' in _ex.get(i, {}) for i, p in human_of.items())
    check('held-out dataset holds your priority_human labels', DS_LABELS_OK, 'if FAIL: delete ' + HELDOUT_DATASET + ' in LangSmith and run again')
    if DS_LABELS_OK:
        held_res = client.evaluate(
            triage, data=HELDOUT_DATASET, evaluators=HELD_EVALUATORS, experiment_prefix=HELD_PREFIX,
            description='R1-05 official held-out run, prompt ' + PROMPT_VERSION + ' (frozen), priority vs human labels', max_concurrency=1,
            metadata={'prompt_version': PROMPT_VERSION, 'model': MODEL, 'temperature': TEMPERATURE, 'set': 'heldout', 'priority_truth': 'priority_human'},
        )
        HELD_EXP = held_res.experiment_name
    else:
        HELD_EXP = None
else:
    HELD_EXP = None
    print('Held-out run skipped: PROMPT_FROZEN is False, or the ingest/prompt check failed.')

held_out = load_experiment(HELD_EXP, n_keys=len(HELD_KEYS)) if HELD_EXP else pd.DataFrame()
if not held_out.empty:
    ref = held_df[['ticket_id', 'queue', 'priority', 'language']].rename(columns={'ticket_id': 'email_id', 'priority': 'tobi_priority'})
    ref['priority_human'] = ref['email_id'].map(human_of)
    held_table = ref.merge(held_out, on='email_id', how='left')
    print('Experiment:', HELD_EXP)
    display(held_table[['email_id', 'language', 'queue', 'queue_ai', 'priority_human', 'tobi_priority', 'priority_ai', 'policy_sections', 'latency_s'] + [k for k in HELD_KEYS if k in held_table]])

In [ ]:
# 3h Human review sheet for Ayhan: langsmith/human_review.csv (held-out drafts, score_1_5 left blank)
LS.mkdir(parents=True, exist_ok=True)
def has_scores(p):
    # True if Ayhan already typed scores: then we never overwrite the file
    if not p.exists():
        return False
    s = pd.read_csv(p, dtype=str).get('score_1_5', pd.Series(dtype=str))
    return s.fillna('').str.strip().ne('').any()

if held_out.empty:
    print('PENDING: human_review.csv is written after the held-out run.')
elif has_scores(OUT['human']):
    print('human_review.csv already has your scores, not overwritten.')
else:
    hr = held_out[['email_id', 'draft_reply']].copy()
    hr['score_1_5'] = ''
    hr['note'] = ''
    hr.to_csv(OUT['human'], index=False)
    print('Wrote', OUT['human'].relative_to(REPO), len(hr), 'rows. Fill score_1_5 (1 = unusable, 5 = send as is) and note.')

In [ ]:
# 3h (optional) Upload your filled scores to LangSmith as feedback "human_score" (runs only if UPLOAD_HUMAN_SCORES = True)
if UPLOAD_HUMAN_SCORES and not held_out.empty and OUT['human'].exists():
    filled = pd.read_csv(OUT['human'], dtype=str).fillna('')
    run_of = dict(zip(held_out['email_id'], held_out['run_id']))
    done = 0
    for _, r in filled.iterrows():
        s = r['score_1_5'].strip()
        if s not in {'1', '2', '3', '4', '5'} or r['email_id'] not in run_of:
            continue
        rid = run_of[r['email_id']]
        # skip if this run already has a human_score (no double upload)
        if any(f.key == 'human_score' for f in client.list_feedback(run_ids=[rid])):
            continue
        client.create_feedback(rid, key='human_score', score=int(s), comment=r['note'][:500])
        done += 1
    print('Uploaded human_score for', done, 'drafts.')
else:
    print('Upload skipped (UPLOAD_HUMAN_SCORES = False, or no held-out run yet).')

In [ ]:
# 3i Trace summary: langsmith/trace_summary.csv (one row per experiment; the held-out row feeds M7 and R1-06)
def summarize(df, exp, dataset, run_set):
    if df.empty:
        return None
    n = int(df['error'].isna().sum())   # drafts traced without an error
    if df['cost_usd'].notna().all():
        cost, source = df['cost_usd'].mean(), 'LangSmith run cost'
    elif PRICE_IN_USD_PER_1M is not None and PRICE_OUT_USD_PER_1M is not None:
        cost = (df['tokens_in'] * PRICE_IN_USD_PER_1M + df['tokens_out'] * PRICE_OUT_USD_PER_1M).mean() / 1e6
        source = 'token usage x OpenAI price constants (cell 1)'
    else:
        cost, source = None, 'PENDING: set PRICE_IN/OUT_USD_PER_1M in cell 1'
    eur = round(cost * USD_TO_EUR, 6) if (cost is not None and USD_TO_EUR) else 'PENDING'
    return {
        'run_set': run_set, 'experiment': exp, 'dataset': dataset, 'prompt_version': PROMPT_VERSION, 'model': MODEL,
        'drafts_traced': n, 'avg_latency_s': round(df['latency_s'].mean(), 2),
        'avg_tokens_in': round(df['tokens_in'].mean(), 1), 'avg_tokens_out': round(df['tokens_out'].mean(), 1),
        'avg_cost_usd': round(cost, 6) if cost is not None else 'PENDING', 'cost_source': source,
        'usd_to_eur': USD_TO_EUR if USD_TO_EUR else 'PENDING', 'usd_to_eur_source': USD_TO_EUR_SOURCE,
        'avg_cost_eur': eur,
        'source': 'LangSmith experiment ' + exp + ' (R1-05, measured)', 'emails': 'Tobi-Bueck tickets (synthetic)',
    }

rows = [summarize(held_out, HELD_EXP, HELDOUT_DATASET, 'heldout'), summarize(dev_out, DEV_EXP, DEV_DATASET, 'dev')]
trace_summary = pd.DataFrame([r for r in rows if r])
if not trace_summary.empty:
    trace_summary.to_csv(OUT['trace'], index=False)
    print('Wrote', OUT['trace'].relative_to(REPO))
    display(trace_summary)
else:
    print('No experiment yet, trace_summary.csv not written.')

In [ ]:
# 3j Monitoring note: langsmith/monitoring_note.md (all numbers come from the experiments above)
SHOTS = LS / 'screenshots'
shots = sorted(p for p in SHOTS.glob('*') if p.suffix.lower() in {'.png', '.jpg', '.jpeg'}) if SHOTS.exists() else []
pct = lambda df, k: (str(int(round(df[k].mean() * 100))) + '%') if (not df.empty and k in df) else 'n/a'
ts = {r['run_set']: r for r in trace_summary.to_dict('records')} if not trace_summary.empty else {}
main = ts.get('heldout') or ts.get('dev') or {}

L = []
add = L.append
add('# LangSmith monitoring note: R1-05')
add('')
add('Prompt ' + PROMPT_VERSION + ' (n8n/triage_prompt.md), model ' + MODEL + ', temperature ' + str(TEMPERATURE) + '. Emails: Tobi-Bueck tickets (synthetic). Written by notebooks/R1-05_langsmith.ipynb.')
add('')
add('## What was monitored')
add('- Every AI triage + draft call is traced in LangSmith as "' + TRACE_NAME + '": the email in, the JSON out (queue, priority, reason, draft reply, policy sections), time, tokens and cost.')
add('- Dev set: ' + DEV_DATASET + ' (10 tuning emails, seed 42), experiment ' + str(DEV_EXP) + '.')
add('- Official held-out set: ' + HELDOUT_DATASET + ' (the 10 timing-test emails, never used for tuning), experiment ' + str(HELD_EXP or 'PENDING (prompt not frozen yet)') + '. Run once per prompt version.')
add('- Dev policy set: ' + POLICY_DATASET + ' (8 synthetic Spreekorb emails, 4 de / 4 en, incl. a deletion request and a legal threat), experiment ' + str(POL_EXP) + '.')
add('- Code checks only, no AI grading. A person scores every held-out draft 1 to 5 (human_score).')
add('')
add('## Ground truth')
add('- Queue truth = the Tobi queue label.')
add('- Priority truth (official) = human labels by Ayhan per Spreekorb Policy 8 (langsmith/heldout_priority_labels.csv), set without seeing AI outputs.')
add('- Tobi priority labels are synthetic and noisy; they are reported only as a secondary column (tobi_priority_match). On the dev set, priority is still compared with Tobi.')
add('')
add('## Results per evaluator (share of drafts passing)')
add('')
add('| Evaluator | What it checks | Dev (10) | Dev policy (8) | Held-out (10) |')
add('|---|---|---|---|---|')
WHAT = {
    'queue_match': 'queue equals the Tobi queue', 'priority_match': 'priority equals the truth (held-out: human label)',
    'priority_within_one': 'priority at most one step off (held-out: human label)', 'no_banned_phrase': 'no banned phrase (EN/DE guardrails) and no em dash',
    'cited_sections_valid': 'cited sections are Policy 1 to 10', 'policy_fact_cited': 'a draft with days, hours or EUR cites a policy section',
    'language_match': 'draft in the email language', 'word_count_60_120': '60 to 120 words (40 to 80 for escalation holding replies)',
    'tobi_priority_match': 'secondary: priority equals the synthetic Tobi label', 'citations_used': 'draft cites at least one policy section',
}
for k in EVAL_KEYS + ['tobi_priority_match', 'citations_used']:
    add('| ' + k + ' | ' + WHAT[k] + ' | ' + pct(dev_out, k) + ' | ' + pct(pol_out, k) + ' | ' + pct(held_out, k) + ' |')
add('')
add('citations_used on non-escalation policy cases: ' + str(CITE_SHARE) + ' (expected high; escalation holding replies may cite nothing).')
add('')
add('Time and cost per draft (' + str(main.get('run_set', 'n/a')) + '): ' + str(main.get('drafts_traced', 'n/a')) + ' drafts traced, average ' + str(main.get('avg_latency_s', 'n/a')) + ' s, ' + str(main.get('avg_tokens_in', 'n/a')) + ' tokens in / ' + str(main.get('avg_tokens_out', 'n/a')) + ' out, ' + str(main.get('avg_cost_usd', 'n/a')) + ' USD (' + str(main.get('cost_source', 'n/a')) + '), ' + str(main.get('avg_cost_eur', 'n/a')) + ' EUR. Full table: langsmith/trace_summary.csv.')
add('')
add('Human score: see langsmith/human_review.csv (feedback "human_score" in LangSmith once uploaded).')
add('')
add('## What it shows about transparency (for Chleo)')
add('- Every draft is traceable: anyone with access can open the trace and see the exact email, the prompt version, the policy sections the AI relied on, and what it wrote.')
add('- Rules are checked by code, the same way every time: banned phrases, length, language, valid policy citations.')
add('- Time and cost are measured per draft, not guessed. They feed the business case (R1-06).')
add('- The AI only drafts. A person reads, edits and decides; the human score records how useful the draft was.')
add('- A prompt change gets a new version and a new experiment, so results stay comparable over time. After any look at held-out results, a prompt change needs a new version.')
add('- Human scores rate AI drafts, never agents. They are not used for appraisals or performance monitoring (draft works agreement).')
add('')
add('## Limits')
add('- 10 emails per set is a small sample: a demo of the method, not proof of quality.')
add('- The emails are synthetic Tobi-Bueck tickets (largely IT support), not Spreekorb emails, and their labels can be noisy. The 8 policy cases are synthetic and written for this test.')
add('- Code checks test form, not truth: a draft can pass all 8 and still be wrong. That is why a person reviews every draft.')
add('- The banned phrase check finds exact phrases only, not close variations, and it also flags "Unfortunately that is not possible" when an alternative follows.')
add('- Language is guessed from common words; time includes the network from the laptop that ran the notebook.')
add('- Cost comes from LangSmith in USD; EUR uses the rate in trace_summary.csv.')
add('- Only synthetic emails were sent. OpenAI and LangSmith are US processors: real customer emails need a data processing agreement, a transfer basis (EU-US Data Privacy Framework or SCCs) and an EU region where offered, first.')
add('- Retention: set a trace retention period in LangSmith in line with the works agreement draft (90 days), or delete the experiments after the capstone.')
add('- No agent names or IDs are recorded. The works agreement is a draft, so this setup is prepared for compliance, not proof of it.')
add('')
add('## Evidence')
add('- LangSmith share link: ' + LANGSMITH_SHARE_URL)
if shots:
    for p in shots:
        add('- ![' + p.stem + '](screenshots/' + p.name.replace(' ', '%20') + ')')
else:
    add('- PENDING screenshot: experiment table with the 8 evaluator columns (langsmith/screenshots/)')
    add('- PENDING screenshot: one trace showing input, output, latency, tokens and cost (langsmith/screenshots/)')
OUT['note'].write_text('\n'.join(L) + '\n', encoding='utf-8')
print('Wrote', OUT['note'].relative_to(REPO), len(L), 'lines')

In [ ]:
# 4 Verify: one PASS/FAIL per acceptance item
held_ok = not held_out.empty
check('INGEST and PROMPT match (kb_compiled.md and n8n)', INGEST_OK and PROMPT_OK)
check('dev experiment has 10 drafts traced', not dev_out.empty and len(dev_out) == 10, DEV_EXP)

# spec 1: 10 items traced with input, output, time and cost
if held_ok:
    full = held_out['email_id'].notna() & held_out['draft_reply'].notna() & held_out['latency_s'].notna() & held_out['tokens_in'].notna()
    check('spec: 10 held-out items traced with input, output, time, tokens', len(held_out) == 10 and full.all() and held_out['error'].isna().all(), HELD_EXP)
    _h = trace_summary[trace_summary['run_set'] == 'heldout'].iloc[0]
    check('spec: cost per item known (LangSmith cost or price constants)', _h['avg_cost_usd'] != 'PENDING', _h['cost_source'])
    check('all evaluator scores (8 + tobi_priority_match) on every held-out run', all(k in held_out and held_out[k].notna().all() for k in HELD_KEYS))
    check('held-out priority compared with priority_human (all 10 labels filled)', LABELS_OK)
else:
    check('spec: 10 held-out items traced (fill priority_human, then PROMPT_FROZEN = True)', False, 'PENDING')

# policy cases and label sheet
pc = pd.read_csv(OUT['policy'], dtype=str).fillna('') if OUT['policy'].exists() else pd.DataFrame()
check('synthetic_policy_cases.csv: 8 rows P01-P08, 4 de + 4 en, synthetic=True, columns', list(pc.columns) == POLICY_COLS and list(pc['email_id']) == ['P0' + str(i) for i in range(1, 9)] and (pc['language'].value_counts().to_dict() == {'de': 4, 'en': 4}) and pc['synthetic'].str.lower().eq('true').all())
check('dev policy experiment has 8 drafts traced', not pol_out.empty and len(pol_out) == 8, POL_EXP)
check('citations_used: >= 80% of non-escalation policy drafts cite a section (threshold set by this notebook)', CITE_SHARE is not None and CITE_SHARE >= 0.8, CITE_SHARE)
lb = pd.read_csv(OUT['labels'], dtype=str) if OUT['labels'].exists() else pd.DataFrame()
check('heldout_priority_labels.csv: columns, holds the 10 held-out emails, no AI output', list(lb.columns) == LABEL_COLS and set(held_df['ticket_id']) <= set(lb.get('email_id', [])))

# spec 2: short note on transparency
note = OUT['note'].read_text(encoding='utf-8') if OUT['note'].exists() else ''
check('spec: monitoring_note.md has transparency, results, limits, evidence', all(s in note for s in ['## What was monitored', '## Results per evaluator', '## What it shows about transparency', '## Limits', 'share link']))
check('monitoring note is short (under 70 lines)', 0 < len(note.splitlines()) < 70, str(len(note.splitlines())) + ' lines')

# spec 3: real cost and time per draft handed to R1-06
ts_cols = ['drafts_traced', 'avg_latency_s', 'avg_tokens_in', 'avg_tokens_out', 'avg_cost_usd', 'avg_cost_eur']
tsr = pd.read_csv(OUT['trace']) if OUT['trace'].exists() else pd.DataFrame()
check('trace_summary.csv has the columns for M7 and R1-06', all(c in tsr for c in ts_cols))
h = tsr[tsr['run_set'] == 'heldout'] if 'run_set' in tsr else tsr.iloc[0:0]   # pick the held-out row by name
check('spec: held-out time per draft measured', len(h) == 1 and int(h['drafts_traced'].iloc[0]) == 10 and float(h['avg_latency_s'].iloc[0]) > 0)
check('spec: cost per draft in EUR set (fill USD_TO_EUR, else PENDING)', len(h) == 1 and str(h['avg_cost_eur'].iloc[0]) != 'PENDING', h['avg_cost_eur'].iloc[0] if len(h) else 'no held-out row')

# human review sheet
hr = pd.read_csv(OUT['human'], dtype=str) if OUT['human'].exists() else pd.DataFrame()
check('human_review.csv: email_id, draft_reply, score_1_5, note, 10 rows', list(hr.columns) == ['email_id', 'draft_reply', 'score_1_5', 'note'] and len(hr) == 10)
check('no key value printed or saved in langsmith/', not any(k in (OUT[x].read_text(encoding='utf-8') if OUT[x].exists() else '') for x in OUT for k in ['sk-', 'lsv2_']))

In [ ]:
# 5 Unchanged: files we do not own keep the same hash (knowledgebase, n8n, data/processed, specs, board.md, other notebooks ...)
AFTER = snapshot()
changed = sorted(k for k in BEFORE if AFTER.get(k) != BEFORE[k])   # a missing file counts as changed
check('knowledgebase/ and n8n/ unchanged', not any(k.startswith(('knowledgebase', 'knowledge_base', 'n8n')) for k in changed))
check('data/processed/ unchanged', not any(k.startswith('data/processed') for k in changed))
check('no other watched file changed', changed == [], str(changed) if changed else 'ok, ' + str(len(AFTER)) + ' files compared')
print()
print(sum(ok for _, ok in results), 'of', len(results), 'checks PASS')

In [ ]:
# 6 Undo (only if RUN_UNDO = True): restore backups, delete output files that did not exist before
if RUN_UNDO:
    for key, p in OUT.items():
        if existed[key]:
            shutil.copy2(BK / p.relative_to(REPO), p)
        else:
            p.unlink(missing_ok=True)
    check('undo', all(p.exists() == existed[k] for k, p in OUT.items()), 'outputs restored or removed')
    print('LangSmith datasets and experiments are not deleted here. Delete them in the LangSmith UI if needed.')
else:
    print('Undo skipped (RUN_UNDO = False)')

In [ ]:
# 7 Git: print only, never run. Ayhan commits.
print('git status')
print('git add notebooks/R1-05_langsmith.ipynb langsmith/ data/processed/synthetic_policy_cases.csv board.md')
print('git commit -m "R1-05 LangSmith sample notebook, trace summary and monitoring note"')
print('git push')